# 05 · Planar structure, used as a test oracle

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/james-kidd/pbn-gradio/blob/main/demos/notebooks/05_topology_invariants.ipynb)

A segmentation into connected regions is a **planar subdivision**, so its
adjacency graph is the dual of that subdivision and therefore planar.

That is not decoration. Planarity is a strong constraint, and it buys
three concrete things: invariants that catch bugs, a complexity
guarantee, and a bound on how many parallel work rounds any plan needs.

The middle section deliberately feeds the checks broken input, because
**an invariant that can never fail is decoration.**

In [1]:
# Bootstrap: clones what it needs in Colab, finds what exists locally.
import pathlib, subprocess, sys

REPO = "https://github.com/james-kidd/pbn-gradio.git"
BRANCH = "main"

def _demos_root():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (base / "nbtools" / "bootstrap.py").exists():
            return base
    dest = (pathlib.Path("/content/pbn-gradio") if pathlib.Path("/content").is_dir()
            else pathlib.Path.cwd() / "pbn-gradio")
    if not (dest / "demos" / "nbtools").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--quiet",
                        "--branch", BRANCH, REPO, str(dest)], check=True)
    return dest / "demos"

sys.path.insert(0, str(_demos_root()))
from nbtools.bootstrap import setup

tk = setup()                           # installs anything missing, first
from nbtools import scenes, display    # these need cv2/matplotlib, so: after

toolkit   /home/user/pbn-gradio  (found locally)
modules   geo, land_use, optimize, pipeline, segmenters, siting, topology
gradio    stubbed (UI layer not needed for these demos)


In [2]:
image = scenes.fragmented()
working = tk.pipeline.load_and_resize(image, 800)
lab = tk.pipeline.preprocess_rgb(working)
MIN_AREA = int(working.shape[0] * working.shape[1] * 0.001)

seg = tk.segmenters.segment(lab, "slic", merge_threshold=8.0, min_area=MIN_AREA,
                            fill=True, n_segments=1200, compactness=10.0, sigma=2.0)
labels = seg.labels
graph = tk.topology.build_region_graph(labels)
print(seg.describe())

slic → merge<8 → drop<480px → fill-gaps (93 regions)


## 1 · The invariants

A simple planar graph on V ≥ 3 vertices satisfies **E ≤ 3V − 6**, and its
mean degree is strictly below 6. These are *necessary* conditions, so a
violation proves something upstream is broken.

The graph is planar by construction **if** the segmentation is well
formed — which means the check is really asking whether the segmentation
is well formed, and that is the question worth asking.

In [3]:
checks = tk.topology.check_planarity(graph, labels)
display.table(
    [["E <= 3V - 6", f"<= {checks['edge_bound']}", checks["E"],
      "pass" if checks["edge_bound_holds"] else "FAIL"],
     ["Mean degree", "< 6", f"{checks['mean_degree']:.2f}",
      "pass" if checks["degree_bound_holds"] else "FAIL"],
     ["Regions connected", "all", f"{len(checks['disconnected_regions'])} violations",
      "pass" if not checks["disconnected_regions"] else "FAIL"]],
    ["Invariant", "Required", "Actual", "Result"])

print(f"V = {checks['V']}   E = {checks['E']}   "
      f"F = {checks['faces']}   (Euler: V - E + F = 2)")

| Invariant | Required | Actual | Result |
|---|---|---|---|
| E <= 3V - 6 | <= 273 | 230 | pass |
| Mean degree | < 6 | 4.95 | pass |
| Regions connected | all | 1 violations | FAIL |

V = 93   E = 230   F = 139   (Euler: V - E + F = 2)


## 2 · Do the checks have teeth?

Two deliberately broken inputs. If these passed, the checks above would
mean nothing.

**Case A — a label covering two disconnected blobs.** This is the failure
that matters in practice: the graph stops being a planar dual, and every
adjacency-derived feature quietly means something other than it claims.

In [4]:
import numpy as np

good = np.zeros((90, 90), np.int32)
for i in range(3):
    for j in range(3):
        good[i * 30:(i + 1) * 30, j * 30:(j + 1) * 30] = i * 3 + j + 1

broken = good.copy()
broken[broken == 9] = 1      # label 1 now covers two separate corners

for name, arr in (("well-formed grid", good), ("split label", broken)):
    g = tk.topology.build_region_graph(arr)
    c = tk.topology.check_planarity(g, arr)
    verdict = ("well formed" if c["well_formed"]
               else f"CAUGHT - disconnected: {c['disconnected_regions']}")
    print(f"{name:18s} V={c['V']:2d} E={c['E']:2d}  ->  {verdict}")

well-formed grid   V= 9 E=12  ->  well formed
split label        V= 8 E=12  ->  CAUGHT - disconnected: [1]


**Case B — K₅, the canonical non-planar graph.** Five mutually adjacent
regions cannot be drawn in the plane, so this is impossible to produce
from a real image. Constructing the graph directly proves the edge bound
actually bites.

In [5]:
k5 = tk.topology.RegionGraph(
    vertices=[1, 2, 3, 4, 5],
    edges={(a, b) for a in range(1, 6) for b in range(a + 1, 6)})
k5.adjacency = {v: {u for u in range(1, 6) if u != v} for v in range(1, 6)}

c = tk.topology.check_planarity(k5, np.ones((4, 4), np.int32))
print(f"K5:  V={c['V']}  E={c['E']}  bound 3V-6={c['edge_bound']}  ->  "
      f"{'CAUGHT' if not c['edge_bound_holds'] else 'passed (wrong!)'}")

K5:  V=5  E=10  bound 3V-6=9  ->  CAUGHT


Both broken inputs are caught. The checks are an oracle, not ceremony.

## 3 · A complexity guarantee, not an observation

`E ≤ 3V − 6` means the adjacency graph has **O(V) edges, not O(V²)**.
Every graph algorithm downstream is therefore linear in the number of
regions — by consequence of planarity, not by luck of these particular
images.

In [6]:
display.table(
    [[checks["V"], checks["E"], checks["V"] ** 2,
      f"{checks['E'] / checks['V']:.2f}", f"{checks['E'] / checks['V']**2:.4f}"]],
    ["V", "actual E", "E if dense (V^2)", "E/V", "E/V^2"])

| V | actual E | E if dense (V^2) | E/V | E/V^2 |
|---|---|---|---|---|
| 93 | 230 | 8649 | 2.47 | 0.0266 |

## 4 · Bounded scheduling, via the Four Colour Theorem

Adjacent parcels cannot be worked at the same time — one crew's access
and spoil occupy both. That constraint *is* graph colouring, and since
the graph is planar, **four colours always suffice**: any site plan
executes in at most four parallel rounds, however many parcels it has.

Constructive four-colouring is famously involved, so this colours greedily
in degeneracy order and then *verifies* the result rather than assuming
it. Planar graphs are 5-degenerate, which bounds greedy at six.

In [7]:
order, degeneracy = tk.topology.degeneracy_ordering(graph)
colouring = tk.topology.colour_graph(graph)
valid = tk.topology.verify_colouring(graph, colouring)
used = len(set(colouring.values()))

display.table(
    [["Degeneracy", degeneracy, "planar graphs are <= 5"],
     ["Greedy bound", degeneracy + 1, "degeneracy + 1"],
     ["Colours used", used, "Four Colour Theorem guarantees 4 suffice"],
     ["Colouring valid", "yes" if valid else "NO", "verified, not assumed"]],
    ["Property", "Value", "Note"])

| Property | Value | Note |
|---|---|---|
| Degeneracy | 4 | planar graphs are <= 5 |
| Greedy bound | 5 | degeneracy + 1 |
| Colours used | 4 | Four Colour Theorem guarantees 4 suffice |
| Colouring valid | yes | verified, not assumed |

In [8]:
parcels = sorted(graph.vertices)[:8]
schedule, rounds = tk.topology.work_schedule(graph, parcels)
print(f"{len(parcels)} parcels -> {rounds} parallel round(s)\n")
for r in range(1, rounds + 1):
    members = sorted(p for p, c in schedule.items() if c == r)
    print(f"  round {r}: parcels {members}")

8 parcels -> 4 parallel round(s)

  round 1: parcels [2, 3]
  round 2: parcels [1, 4, 6, 8]
  round 3: parcels [5]
  round 4: parcels [7]


## 5 · Holes

A region's Euler number is `components − holes`, so a parcel wrapped
around a pond reports one hole. That is a planning fact, not a curiosity:
interior boundary cannot be built on and has to be worked around.

In [9]:
holes = tk.topology.region_holes(labels)
holed = {k: v for k, v in holes.items() if v}
print(f"{len(holed)} of {len(holes)} regions contain holes")
if holed:
    display.table(sorted(holed.items())[:10], ["region", "holes"])

3 of 93 regions contain holes


| region | holes |
|---|---|
| 5 | 1 |
| 67 | 1 |
| 83 | 1 |